# Oxford iSeg + U-Net + W&B

このNotebookは、`torch-foundry`の学習関数をKaggleから再利用する例です。Weights & Biases（W&B）には、エポックごとの損失・IoU・Diceと、学習後の予測比較画像を保存します。

## 1. リポジトリと依存パッケージの準備

Kaggle NotebookのInternet設定を有効にしてから実行してください。

In [ ]:
![ -d /kaggle/working/torch-foundry/.git ] || git clone https://github.com/Kaz0818/torch-foundry.git /kaggle/working/torch-foundry
%cd /kaggle/working/torch-foundry
!pip install -e .

## 2. W&Bの認証と保存先

KaggleのAdd-ons > Secretsへ`WANDB_API_KEY`を登録してください。`WANDB_PROJECT`は保存先のW&Bプロジェクト名なので、実行前に必ず変更してください。`WANDB_RUN_NAME`は任意です。

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

WANDB_PROJECT = ""  # 必須: 利用先のW&Bプロジェクト名
WANDB_RUN_NAME = None  # 任意: 例 "baseline-64px"

if not WANDB_PROJECT:
    raise ValueError("WANDB_PROJECTを設定してください")

## 3. データ、モデル、保存先の準備

この例は短い動作確認のため、Oxford iSegの先頭20組を使います。設定値やモデルは、このセルを編集して変更できます。

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import optim

from vision.segmentation.config import Config
from vision.segmentation.datasets.dataset import get_dataloader
from vision.segmentation.datasets.oxford_iseg import prepare_oxford_iseg
from vision.segmentation.losses import build_loss
from vision.segmentation.models.unet import UNet
from vision.segmentation.training.train import train
from vision.segmentation.utils.visualization import get_device, plot_overlay

PROJECT_ROOT = Path.cwd()
config = Config.from_json(PROJECT_ROOT / "vision/segmentation/config.json")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

data_root = Path(config.data_root)
if not data_root.is_absolute():
    data_root = PROJECT_ROOT / data_root

images, masks = prepare_oxford_iseg(data_root)
images, masks = images[:20], masks[:20]
train_loader, val_loader = get_dataloader(
    images=images,
    masks=masks,
    batch_size=config.batch_size,
    image_size=config.image_size,
    val_ratio=config.val_ratio,
    seed=config.seed,
)

device = get_device()
model = UNet(3, config.num_classes).to(device)
criterion = build_loss(config)
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
total_parameters = sum(parameter.numel() for parameter in model.parameters())

output_dir = (
    PROJECT_ROOT
    / "vision/segmentation/runs"
    / datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
)
output_dir.mkdir(parents=True, exist_ok=False)

effective_config = config.to_dict()
effective_config["data_root"] = str(data_root)
(output_dir / "config.json").write_text(
    json.dumps(effective_config, indent=2) + "\n", encoding="utf-8"
)

run_config = {
    **effective_config,
    "dataset_name": "Oxford iSeg",
    "model_name": "UNet",
    "train_samples": len(train_loader.dataset),
    "val_samples": len(val_loader.dataset),
    "total_parameters": total_parameters,
    "device": str(device),
}

print("device:", device)
print("output directory:", output_dir)

## 4. 学習とW&Bへの保存

`train()`はW&Bに依存していません。W&Bの`run.log`を`metric_logger`へ渡した場合だけ、エポックごとの指標を保存します。

In [ ]:
with wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config=run_config,
    dir=str(output_dir),
) as run:
    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    history = train(
        train_loader=train_loader,
        val_loader=val_loader,
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=config.num_epochs,
        num_classes=config.num_classes,
        output_dir=output_dir,
        metric_logger=run.log,
    )

    val_images, val_masks = next(iter(val_loader))
    model.eval()
    with torch.inference_mode():
        val_logits = model(val_images.to(device))
        val_predictions = val_logits.argmax(dim=1)

    comparison = plot_overlay(
        val_images, val_masks, val_predictions, max_images=5
    )
    try:
        run.log({"results/predictions": wandb.Image(comparison)})
    finally:
        plt.close(comparison)

    run_url = run.url

print("saved epochs:", len(history["train_loss"]))
print("local artifacts:", output_dir)
print("W&B run:", run_url)